In [ ]:
# correct version with both methods and image input with cached info V3
from google import generativeai as genai

from flask import Flask, request, jsonify, render_template_string
from io import BytesIO
import base64
import requests
from pyngrok import ngrok
import json
import traceback
import re
from PIL import Image


# Initialize Gemini client
genai.configure(api_key="enter_your_api")
app = Flask(__name__)
html_template = """
<!DOCTYPE html>
<html>
<head>
    <title>Nutrition & Food Analysis</title>
    <style>
        body { font-family: Arial; padding: 30px; max-width: 1200px; margin: auto; }
        .top-section { margin-bottom: 40px; }
        .row { display: flex; gap: 40px; flex-wrap: wrap; }
        .method { flex: 1 1 500px; min-width: 400px; }
        textarea, input[type=file] { width: 100%; font-size: 14px; margin-top: 10px; }
        button { padding: 10px 20px; margin-top: 10px; font-size: 16px; }
        pre, p, table {
            background: #f4f4f4;
            padding: 10px;
            border-radius: 6px;
            white-space: pre-wrap;    /* ✅ wraps long lines */
            word-wrap: break-word;    /* ✅ breaks long words */
            overflow-x: auto;
        }
        img {
            margin-top: 10px;
            max-width: 100%;
            max-height: 300px;
            object-fit: contain;
            border-radius: 8px;
            display: block;
        }
        table {
            width: 100%;
            max-width: 100%;
            border-collapse: collapse;
            table-layout: fixed;
            word-wrap: break-word;
        }
        td, th {
            border: 1px solid #ccc;
            padding: 8px;
            text-align: left;
            font-size: 14px;
        }
        .table-wrapper {
            overflow-x: auto;
            max-width: 100%;
        }
        .spinner {
            border: 4px solid #f3f3f3;
            border-top: 4px solid #3498db;
            border-radius: 50%;
            width: 30px;
            height: 30px;
            animation: spin 1s linear infinite;
            margin: 20px auto;
            display: none;
        }
        @keyframes spin {
            0% { transform: rotate(0deg); }
            100% { transform: rotate(360deg); }
        }
    </style>
</head>
<body>

    <!-- 🔼 Cached Image Gallery Section -->
    <div class="top-section">
        <h2>Select a Cached Food Image</h2>
        <div id="thumbnailGallery" style="display: flex; flex-wrap: wrap; gap: 12px; margin-bottom: 20px;"></div>

        <h3>Selected Image:</h3>
        <img id="originalImageTop" style="display:none" />
    </div>
    <div class="top-section">
      <h2>Upload Your Own Food Image</h2>
      <input type="file" id="uploadInput" accept="image/*">
      <button onclick="uploadImage()">Analyze Uploaded Image</button>
      <div id="spinnerUpload" class="spinner"></div>
    </div>

    <!-- 🔽 Result Comparison Section -->
    <div class="row">
    <!-- Method 1 -->
    <div class="method">
        <h2>Method 1: Segmentation Method</h2>

        <h3>Original Image Result:</h3>
        <div class="table-wrapper" id="m1InitialInfo"></div>

        <h3>Gemini Regenerated Image:</h3>
        <img id="geminiImageM1" style="display:none" />

        <h3>New Image Result:</h3>
        <div class="table-wrapper" id="m1NewInfo"></div>

        <h3>Information Comparison:</h3>
        <div class="table-wrapper" id="m1Table"></div>
    </div>

    <!-- Method 2 -->
    <div class="method">
        <h2>Method 2: Gemini Detection Method</h2>

        <h3>Original Image Result:</h3>
        <div class="table-wrapper" id="m2InitialInfo"></div>

        <h3>Gemini Regenerated Image:</h3>
        <img id="geminiImageM2" style="display:none" />

        <h3>New Image Result:</h3>
        <div class="table-wrapper" id="m2NewInfo"></div>

        <h3>Information Comparison:</h3>
        <div class="table-wrapper" id="m2Table"></div>
    </div>

    <div id="spinnerUpload" class="spinner"></div>
</div>


    <script>
        // Load cached thumbnails on page load
        async function loadGallery() {
            const res = await fetch("/list_images");
            let data;
            try {
                data = await res.json();
            } catch (err) {
                console.error("🛑 Failed to parse /list_images response:", await res.text());
                return;
            }

            if (!data.image_list) {
                console.warn("No image list found:", data);
                return;
            }

            const gallery = document.getElementById("thumbnailGallery");
            gallery.innerHTML = "";  // Clear any existing

            data.image_list.forEach(filename => {
                const img = document.createElement("img");
                img.src = "/static/test_images/" + filename;
                img.alt = filename;
                img.title = filename;
                img.style = "width:100px; cursor:pointer; border-radius:6px; border:1px solid #ccc;";
                img.onclick = () => loadImageData(filename);
                gallery.appendChild(img);
            });

            console.log("🟢 loadGallery() completed");
        }


        // Load JSON content on thumbnail click
       async function loadImageData(filename) {
          const res = await fetch("/get_cached_data?filename=" + filename);
          const data = await res.json();

          document.getElementById("originalImageTop").src = "/static/test_images/" + filename;
          document.getElementById("originalImageTop").style.display = "block";

          // ---------------- Nutrition Table ----------------
          function buildNutritionTable(methodData, method) {
              const initial = methodData?.initial_info || {};
              const updated = methodData?.new_info || {};

              const labelMap = {
                  "Calories": "Calories (kcal)",
                  "Protein": "Protein (g)",
                  "Fat": "Fat (g)",
                  "Carbohydrates": "Carbohydrates (g)",
                  "Fiber": "Fiber (g)",
                  "Sugar": "Sugar (g)",
                  "Sodium": "Sodium (mg)"
              };

              let tableHTML = "<table><tr><th>Metric</th><th>Initial</th><th>Updated</th></tr>";

              if (method === "method1") {
                  const keyMap = {
                      "Calories": "total_energy_kcal",
                      "Protein": "total_protein_kcal",
                      "Fat": "total_fat_kcal",
                      "Carbohydrates": "total_carbohydrate_kcal",
                      "Fiber": "total_fiber_kcal",
                      "Sugar": "total_sugar_kcal",
                      "Sodium": "total_sodium_kcal"
                  };
                  for (const [label, display] of Object.entries(labelMap)) {
                      const key = keyMap[label];
                      const v1 = initial[key] ?? "-";
                      const v2 = updated[key] ?? "-";
                      tableHTML += `<tr><td>${display}</td><td>${v1}</td><td>${v2}</td></tr>`;
                  }
              } else if (method === "method2") {
                  const parsed1 = initial.parsed_nutrition || {};
                  const parsed2 = updated.parsed_nutrition || {};
                  for (const [label, display] of Object.entries(labelMap)) {
                      const v1 = parsed1[label]?.["Quantity Number/Value"] ?? "-";
                      const v2 = parsed2[label]?.["Quantity Number/Value"] ?? "-";
                      tableHTML += `<tr><td>${display}</td><td>${v1}</td><td>${v2}</td></tr>`;
                  }
              }

              tableHTML += "</table>";
              return tableHTML;
          }

          // ---------------- Pixel Coverage Table (Method 1) ----------------
          function buildPixelPercentageTable(methodData) {
              const initial = methodData?.initial_info?.pixel_percentages || {};
              const updated = methodData?.new_info?.pixel_percentages || {};
              const allLabels = new Set([...Object.keys(initial), ...Object.keys(updated)]);

              let html = "<table><tr><th>Label</th><th>Original %</th><th>Updated %</th></tr>";
              Array.from(allLabels).sort().forEach(label => {
                  const v1 = initial[label] ?? "NA";
                  const v2 = updated[label] ?? "NA";
                  html += `<tr><td>${label}</td><td>${v1}</td><td>${v2}</td></tr>`;
              });
              html += "</table>";
              return html;
          }

          // ---------------- Ingredient Comparison Table (Method 2) ----------------
          function buildIngredientComparisonTable(methodData) {
              function parse(text) {
                  const lines = text.trim().split("\\n");
                  const result = {};
                  for (const line of lines) {
                      const parts = line.split("|").map(p => p.trim());
                      if (parts.length === 4) {
                          const name = parts[0].toLowerCase();
                          const qty = parts[1] + " " + parts[2];
                          result[name] = qty;
                      }
                  }
                  return result;
              }

              const desc1 = methodData?.initial_info?.image_description || "";
              const hidden1 = methodData?.initial_info?.hidden_ingredients || "";
              const desc2 = methodData?.new_info?.image_description || "";
              const hidden2 = methodData?.new_info?.hidden_ingredients || "";

              const ingr1 = parse(desc1 + "\\n" + hidden1);
              const ingr2 = parse(desc2 + "\\n" + hidden2);
              const allIngredients = new Set([...Object.keys(ingr1), ...Object.keys(ingr2)]);

              let html = "<table><tr><th>Ingredient</th><th>Original</th><th>Updated</th></tr>";
              Array.from(allIngredients).sort().forEach(name => {
                  html += `<tr><td>${name}</td><td>${ingr1[name] || "NA"}</td><td>${ingr2[name] || "NA"}</td></tr>`;
              });
              html += "</table>";
              return html;
          }

          // ---------------- Method 1 Display ----------------
          const m1 = data.method1 || {};
          document.getElementById("m1InitialInfo").innerHTML =
              '<pre>' + JSON.stringify(m1.initial_info || {}, null, 2) + '</pre>';

          document.getElementById("geminiImageM1").src = m1.gemini_generated_image
              ? "/gemini_generated/" + m1.gemini_generated_image.split("/").pop()
              : "";
          document.getElementById("geminiImageM1").style.display = m1.gemini_generated_image ? "block" : "none";

          document.getElementById("m1NewInfo").innerHTML =
              '<pre>' + JSON.stringify(m1.new_info || {}, null, 2) + '</pre>';

          document.getElementById("m1Table").innerHTML =
              "<h4>Nutrition Comparison</h4>" + buildNutritionTable(m1, "method1") +
              "<br><h4>Pixel Percentage Comparison</h4>" + buildPixelPercentageTable(m1);

          // ---------------- Method 2 Display ----------------
          const m2 = data.method2 || {};
          document.getElementById("m2InitialInfo").innerHTML =
              '<pre>' + JSON.stringify(m2.initial_info || {}, null, 2) + '</pre>';

          document.getElementById("geminiImageM2").src = m2.gemini_generated_image
              ? "/gemini_generated/" + m2.gemini_generated_image.split("/").pop()
              : "";
          document.getElementById("geminiImageM2").style.display = m2.gemini_generated_image ? "block" : "none";

          document.getElementById("m2NewInfo").innerHTML =
              '<pre>' + JSON.stringify(m2.new_info || {}, null, 2) + '</pre>';

          document.getElementById("m2Table").innerHTML =
              "<h4>Nutrition Comparison</h4>" + buildNutritionTable(m2, "method2") +
              "<br><h4>Ingredient Comparison</h4>" + buildIngredientComparisonTable(m2);
      }

        // Initialize
        document.addEventListener("DOMContentLoaded", loadGallery);

       async function uploadImage() {
          const fileInput = document.getElementById("uploadInput");
          const file = fileInput.files[0];
          const spinner = document.getElementById("spinnerUpload");

          if (!file) {
              alert("Please select an image.");
              return;
          }

          spinner.style.display = "block";

          try {
              const formData = new FormData();
              formData.append("image", file);

              const res = await fetch("/upload_image_for_models", {
                  method: "POST",
                  body: formData
              });

              let data;
              try {
                  data = await res.json();
              } catch (err) {
                  alert("❌ Server error. Check console for details.");
                  console.error("Invalid JSON:", await res.text());
                  return;
              }

              // ✅ Show uploaded image at top
              const originalImage = "data:image/png;base64," + data.base64_image;
              document.getElementById("originalImageTop").src = originalImage;
              document.getElementById("originalImageTop").style.display = "block";

              // ✅ Method 1
              const m1 = data.result_m1 || {};
              const g1 = data.gemini_m1 || {};

              document.getElementById("m1InitialInfo").innerHTML =
                  '<pre>' + JSON.stringify(m1, null, 2) + '</pre>';

              document.getElementById("geminiImageM1").src =
                  "data:image/png;base64," + (g1.base64_image || "");
              document.getElementById("geminiImageM1").style.display =
                  g1.base64_image ? "block" : "none";

              const newResult1 = g1.new_generated_result || {};
              document.getElementById("m1NewInfo").innerHTML =
                  '<pre>' + JSON.stringify(newResult1, null, 2) + '</pre>';

              document.getElementById("m1Table").innerHTML =
                  "<h4>Nutrition Comparison</h4>" + (g1.table_html || "");

              // ✅ Method 2
              const m2 = data.nutrition_text_m2 || {};
              const g2 = data.gemini_m2 || {};

              document.getElementById("m2InitialInfo").innerHTML =
                  '<pre>' + JSON.stringify(m2, null, 2) + '</pre>';

              document.getElementById("geminiImageM2").src =
                  "data:image/png;base64," + (g2.base64_image || "");
              document.getElementById("geminiImageM2").style.display =
                  g2.base64_image ? "block" : "none";

              const newResult2 = g2.new_generated_result || {};
              document.getElementById("m2NewInfo").innerHTML =
                  '<pre>' + JSON.stringify(newResult2, null, 2) + '</pre>';

              document.getElementById("m2Table").innerHTML =
                  "<h4>Nutrition Comparison</h4>" + (g2.table_html || "");

          } catch (err) {
              alert("❌ Unexpected error. Check console.");
              console.error("Upload error:", err);
          } finally {
              spinner.style.display = "none";  // ✅ Always hide spinner at the end
          }
      }


    </script>
</body>

</html>
"""

@app.route("/")
def home():
    return render_template_string(html_template)

@app.route("/generate_image", methods=["POST"])
def generate_image():
    user_input_raw = request.json.get("prompt", "")
    if not user_input_raw:
        return jsonify({"error": "Missing prompt"}), 400

    try:
        user_input = json.loads(user_input_raw)
    except Exception:
        return jsonify({"error": "Invalid JSON format"}), 400

    full_prompt = "Consider you are a nutritionist. Given the following information, generate a real food image. Make sure to generate a Food Image:\n" + user_input_raw

    try:
        model = genai.GenerativeModel("gemini-2.0-flash-preview-image-generation")
        response = model.generate_content(contents=full_prompt,
                                          generation_config={"response_modalities": ["TEXT", "IMAGE"]})

        image_bytes = None
        gemini_text = ""
        for part in response.candidates[0].content.parts:
            if hasattr(part, 'text') and part.text:
                gemini_text += part.text
            elif hasattr(part, 'inline_data'):
                image_bytes = part.inline_data.data

        if image_bytes is None:
            return jsonify({"error": "No image returned from Gemini"}), 500

        base64_image = base64.b64encode(image_bytes).decode("utf-8")

        verification_result = None
        try:
            autodl_response = requests.post(
                "https://d82e-1-27-207-187.ngrok-free.app/receive_image",
                json={"base64_image": base64_image, "prompt": full_prompt},
                timeout=40
            )
            verification_result = autodl_response.json().get("result", {})
        except Exception:
            verification_result = "waiting"

        generated_result = verification_result if isinstance(verification_result, dict) else {}

        # Table 1: Known nutrition metrics
        rows = [
            ("total_energy_kcal", "Energy (kcal)"),
            ("total_protein_g", "Protein (g)"),
            ("total_fat_g", "Fat (g)"),
            ("total_carbohydrate_g", "Carbohydrates (g)"),
            ("total_fiber_g", "Fiber (g)"),
            ("total_sugar_g", "Sugar (g)"),
            ("total_sodium_mg", "Sodium (mg)")
        ]

        table_html = "<table><tr><th>Nutritional Metric</th><th>Original Image Nutrition</th><th>New Image Nutrition</th></tr>"
        for key, label in rows:
            user_val = user_input.get(key, "-")
            model_val = generated_result.get(key, "waiting")
            table_html += f"<tr><td>{label}</td><td>{user_val}</td><td>{model_val}</td></tr>"
        table_html += "</table>"

        # Table 2: Pixel percentages like label coverage
        user_percentages = user_input.get("pixel_percentages", {})
        model_percentages = generated_result.get("pixel_percentages", {})
        all_labels = set(user_percentages.keys()) | set(model_percentages.keys())

        table2_html = "<br><table><tr><th>Ingredient Metric</th><th>Original Image Ingredient Percentage</th><th>New Image Ingredient Percentage</th></tr>"
        for label in sorted(all_labels):
            user_val = user_percentages.get(label, "NA")
            model_val = model_percentages.get(label, "NA")
            table2_html += f"<tr><td>{label}</td><td>{user_val}</td><td>{model_val}</td></tr>"
        table2_html += "</table>"

        return jsonify({
            "base64_image": base64_image,
            "text": gemini_text,
            "table_html": table_html + table2_html,
            "new_generated_result": generated_result
        })

    except Exception:
        return jsonify({"error": traceback.format_exc()}), 500



@app.route("/generate_image_from_text", methods=["POST"])
def generate_image_from_text():
    user_input_raw = request.json.get("prompt", "")
    if not user_input_raw:
        return jsonify({"error": "Missing prompt"}), 400

    full_prompt = (
        "Suppose you are a nutritionist. Here's the information you have. Generate a food image based on these information:\n"
        + json.dumps(user_input_raw, indent=2)
    )

    try:
        model = genai.GenerativeModel("gemini-2.0-flash-preview-image-generation")
        response = model.generate_content(
            contents=full_prompt,
            generation_config={"response_modalities": ["TEXT", "IMAGE"]}
        )

        image_bytes = None
        for part in response.candidates[0].content.parts:
            if hasattr(part, 'inline_data'):
                image_bytes = part.inline_data.data

        if image_bytes is None:
            return jsonify({"error": "No image returned from Gemini"}), 500

        base64_image = base64.b64encode(image_bytes).decode("utf-8")

        try:
            autodl_response = requests.post(
                "http://localhost:5000/analyze_image",
                files={"image": BytesIO(base64.b64decode(base64_image))}
            )
            analysis_result = autodl_response.json()
        except Exception as err:
            return jsonify({"error": f"Image generated but analysis failed: {err}"}), 500

        # Table 1: Nutrition Comparison
        user_nutrition = {}
        user_input_text = user_input_raw.get("nutrition", "")

        # Define display labels with units
        nutrient_labels = {
            "Calories": "Calories (kcal)",
            "Protein": "Protein (g)",
            "Fat": "Fat (g)",
            "Carbohydrates": "Carbohydrates (g)",
            "Fiber": "Fiber (g)",
            "Sugar": "Sugar (g)",
            "Sodium": "Sodium (mg)"
        }

        # Extract user nutrition values
        for line in user_input_text.splitlines():
            for key in nutrient_labels:
                if key in line:
                    try:
                        user_nutrition[key] = line.split("|")[1].strip()
                    except IndexError:
                        user_nutrition[key] = "-"

        # Build nutrition comparison table
        table_html = "<table><tr><th>Nutrition Metric</th><th>Original Image Nutrition</th><th>New Image Nutrition</th></tr>"
        for key, label in nutrient_labels.items():
            model_val = (
                analysis_result.get("nutrition", "").split(f"{key} | ")[1].split("|")[0].strip()
                if f"{key} | " in analysis_result.get("nutrition", "") else "waiting"
            )
            table_html += f"<tr><td>{label}</td><td>{user_nutrition.get(key, '-')}</td><td>{model_val}</td></tr>"
        table_html += "</table>"


        # Table 3: Ingredient + Quantity comparison
        def parse_ingredients(text):
            lines = text.strip().splitlines()
            parsed = []
            for line in lines:
                parts = [p.strip() for p in line.split("|")]
                if len(parts) == 4:
                    name, qty, unit, _ = parts
                    parsed.append((name.lower(), f"{qty} {unit}"))
            return dict(parsed)

        user_ingredients = parse_ingredients(user_input_raw.get("description", "") + "\n" + user_input_raw.get("hidden", ""))
        model_ingredients = parse_ingredients(analysis_result.get("description", "") + "\n" + analysis_result.get("hidden", ""))
        all_ingr = sorted(set(user_ingredients) | set(model_ingredients))

        table3_html = "<br><table><tr><th>Metric</th><th>Original Input</th><th>New Generated Result</th></tr>"
        for ingr in all_ingr:
            user_val = user_ingredients.get(ingr, "NA")
            model_val = model_ingredients.get(ingr, "NA")
            table3_html += f"<tr><td>{ingr}</td><td>{user_val}</td><td>{model_val}</td></tr>"
        table3_html += "</table>"

        return jsonify({
            "base64_image": base64_image,
            "table_html": table_html + table3_html,
            "new_generated_result": analysis_result
        })

    except Exception:
        return jsonify({"error": traceback.format_exc()}), 500


@app.route("/analyze_image", methods=["POST"])
def analyze_image():
    if 'image' not in request.files:
        return jsonify({"error": "No image uploaded"}), 400

    image_file = request.files['image']
    image_bytes = image_file.read()

    try:
        full_prompt = (
            "Describe the food dish in this image.\n"
            "Return the dish name on the first line.\n"
            "Then list each visible ingredient on a new line in the format: Ingredient | Quantity Number | Unit | Reasoning.\n"
            "Quantity Number must be a numeric value only.\n"
            "Avoid vague ranges or approximations like 'a few' or 'some'.\n"
            "Be concise and avoid unnecessary descriptions.\n"
            "Skip any background or utensils."
        )

        image = Image.open(BytesIO(image_bytes)).convert("RGB")

        model = genai.GenerativeModel("gemini-2.0-flash-preview-image-generation")
        response = model.generate_content(
            contents=[full_prompt, image],
            generation_config={"response_modalities": ["TEXT", "IMAGE"]}
        )

        description = "".join(
            part.text for part in response.candidates[0].content.parts if hasattr(part, "text") and part.text
        )
        print("Description:", description)

    except Exception:
        return jsonify({"error": traceback.format_exc()}), 500

    try:
        lines = description.splitlines()
        dish_name = lines[0].strip()
        visible_ingredients = "\n".join(
            [line for line in lines[1:] if '|' in line and len(line.split('|')) == 4]
        )

        hidden_model = genai.GenerativeModel("gemini-2.5-flash")
        hidden_response = hidden_model.generate_content(
            contents=f"""
                You are a recipe analyst.
                For the dish '{dish_name}', given the following visible ingredients:
                {visible_ingredients},
                list only the likely hidden ingredients used in traditional or common recipes for this dish.
                Format each hidden ingredient on a new line like this: Ingredient | Quantity Number | Unit | Reasoning.
                Quantity Number must be a numeric value only.
                Only include core items like oil, butter, sauces, or spices typically used. Avoid optional or garnish ingredients.
                Do NOT use any vague descriptions. Be clear and formatted strictly.
            """,
            generation_config={"response_modalities": ["TEXT"]}
        )
        hidden_text = hidden_response.text
        print("Hidden Ingredients:", hidden_text)

        nutrition_response = hidden_model.generate_content(
            contents=f"""
                You are a nutritionist.
                The user has provided the visible ingredients from a dish named '{dish_name}'.
                Ingredients:
                {visible_ingredients}

                Your task is to output the nutritional breakdown per serving (based on image analysis).
                Output each nutrient on a new line in this exact format:
                Nutrient | Value | Unit | Reasoning
                Value must be a numeric value only.
                Include at least these nutrients: Calories, Protein, Fat, Carbohydrates, Fiber, Sugar, Sodium.
                Be strict with the format.
            """,
            generation_config={"response_modalities": ["TEXT"]}
        )
        nutrition_text = nutrition_response.text
        print("Nutrition:", nutrition_text)

        return jsonify({
            "description": description,
            "hidden": hidden_text,
            "nutrition": nutrition_text
        })

    except Exception:
        return jsonify({"error": traceback.format_exc()}), 500




@app.route("/upload_image_for_models", methods=["POST"])
def upload_image_for_models():
    if 'image' not in request.files:
        return jsonify({"error": "No image uploaded"}), 400

    image_file = request.files['image']
    image_bytes = image_file.read()
    base64_image = base64.b64encode(image_bytes).decode("utf-8")

    try:
        # 🔹 Method 1: External API call to receive_image service
        try:
            autodl_response = requests.post(
                "https://d82e-1-27-207-187.ngrok-free.app/receive_image",
                json={"base64_image": base64_image, "prompt": "Analyze nutrition of this image."},
                timeout=40
            )
            result_m1 = autodl_response.json().get("result", {})
        except Exception:
            result_m1 = {}

        # 🔹 Method 1: Call generate_image
        try:
            generate1_resp = requests.post(
                "http://localhost:5000/generate_image",
                json={"prompt": json.dumps(result_m1)}
            )
            gemini_m1 = generate1_resp.json()
        except Exception:
            gemini_m1 = {}

        # 🔹 Method 2: Local model analysis
        try:
            analyze_response = requests.post(
                "http://localhost:5000/analyze_image",
                files={"image": BytesIO(base64.b64decode(base64_image))}
            )
            nutrition_text_m2 = analyze_response.json()
        except Exception:
            nutrition_text_m2 = {}

        # 🔹 Method 2: Call generate_image_from_text
        try:
            generate2_resp = requests.post(
                "http://localhost:5000/generate_image_from_text",
                json={"prompt": nutrition_text_m2}
            )
            gemini_m2 = generate2_resp.json()
        except Exception:
            gemini_m2 = {}

        return jsonify({
            "base64_image": base64_image,
            "result_m1": result_m1,
            "gemini_m1": gemini_m1,
            "nutrition_text_m2": nutrition_text_m2,
            "gemini_m2": gemini_m2
        })

    except Exception:
        return jsonify({"error": traceback.format_exc()}), 500


import os
from flask import send_from_directory

# Load JSON from disk
with open("test_cached_information.json") as f:
    cached_json = json.load(f)

@app.route("/list_images")
def list_images():
    try:
        image_dir = "test_images"
        image_list = [f for f in os.listdir(image_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        return jsonify({"image_list": image_list})
    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route("/get_cached_data")
def get_cached_data():
    filename = request.args.get("filename", "")
    data = cached_json.get(filename, {})
    return jsonify(data)

# Serve static images (test_images folder must be linked or copied to /static/test_images)
@app.route("/static/test_images/<path:filename>")
def static_images(filename):
    return send_from_directory("test_images", filename)

from flask import send_file


@app.route("/gemini_generated/<path:image_name>")
def serve_gemini_image(image_name):
    model1_path = "generated_images_model1"
    model2_path = "generated_images_model2"

    for base in [model1_path, model2_path]:
        full = os.path.join(base, image_name)
        if os.path.exists(full):
            return send_file(full, mimetype="image/jpeg")

    return "Image not found", 404


# Launch
public_url = ngrok.connect(5000)
print("\U0001F310 Public URL:", public_url)
app.run(host="0.0.0.0", port=5000)


In [ ]:
pip install pyngrok

In [ ]:
!ngrok config add-authtoken 2yhnlhfGvh3MreM3nTmmXM60L3F_6fQKGVBRQayHHSwYcj41r
